In [ ]:
# Configure logger
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("Exploration")

In [ ]:
import geopandas as gpd
import shapely
import numpy as np
from get_and_transform_data import get_and_transform_data
import RiskMap
from pollutant_parameters import P_values, NH4_values

## Obtaining data

In [ ]:
cso_all, wwtw_all, d, rivers_all = get_and_transform_data()

In [ ]:
# Add the catchment geojson url of your choosing here to clip the data to that catchment
cmnt = gpd.read_file(
    "https://environment.data.gov.uk/catchment-planning/OperationalCatchment/3052.geojson",
)  # Browney
# cmnt = gpd.read_file(
#     "https://environment.data.gov.uk/catchment-planning/WaterBody/GB103023075780.geojson"
# )  # Other Catchment

In [ ]:
cmnt["geometry"] = cmnt.make_valid()
cmnt = cmnt[["geometry"]][cmnt.geometry.type == "Polygon"]
cmnt.to_crs(epsg=27700, inplace=True)
cmnt_mask = gpd.GeoDataFrame(
    geometry=[cmnt.geometry.union_all()], crs=cmnt.crs
)

### Clip to Catchment Data

In [ ]:
rivers = gpd.overlay(
    rivers_all, cmnt_mask, how="intersection", keep_geom_type=True
)
cso = gpd.clip(cso_all, cmnt_mask)
wwtw = gpd.clip(wwtw_all, cmnt_mask)

## Construct Risk Info Map

In [ ]:
RIVER_RESOLUTION: float = 300
MAX_OUTFALL_DIST_FROM_RIVER: float = 700
MAX_RISK_DISTANCE: float = np.inf

In [ ]:
from utils.rivers import linear_flow

# Generate Graph from Rivers
riskmap = RiskMap.RiskMap(
    riversdf=rivers,
    id_col="RivID",
    resolution=RIVER_RESOLUTION,
    river_flow_func=lambda x: linear_flow(x["US_Accum"]),
    catchment_mask=cmnt_mask,
)
riskmap.delineate_catchments()

In [ ]:
# Add CSO information
outfallLocations = riskmap.add_point_info_to_map(
    cso.index,  # type: ignore
    cso["geometry"],  # type: ignore
    "csoInfo",
    MAX_RISK_DISTANCE,
    MAX_OUTFALL_DIST_FROM_RIVER,
)
cso = cso.assign(outfallLocation=outfallLocations)

In [ ]:
# Add WwTW information
outfallLocations = riskmap.add_point_info_to_map(
    wwtw.index,  # type: ignore
    wwtw["geometry"],  # type: ignore
    "wwtwInfo",
    MAX_RISK_DISTANCE,
    MAX_OUTFALL_DIST_FROM_RIVER,
)
wwtw = wwtw.assign(outfallLocation=outfallLocations)

In [ ]:
from utils.agricultureRisk import (
    agriculture_land_cover_load,
    agriculture_livestock_load,
)

# Add Agriculture land use load info
riskmap.add_diffuse_info_to_map(
    info_col_name="agri_P_load",
    load_method=agriculture_land_cover_load,
    parameter="P",
)
riskmap.add_diffuse_info_to_map(
    info_col_name="livestock_P_load",
    load_method=agriculture_livestock_load,
    parameter="P",
)
# Add Agriculture land use load info
riskmap.add_diffuse_info_to_map(
    info_col_name="agri_NH4_load",
    load_method=agriculture_land_cover_load,
    parameter="NH4",
)
riskmap.add_diffuse_info_to_map(
    info_col_name="livestock_NH4_load",
    load_method=agriculture_livestock_load,
    parameter="NH4",
)

In [ ]:
from utils.urbanRisk import urban_road, urban_buildings

riskmap.add_diffuse_info_to_map(
    info_col_name="urban_road_P_load",
    load_method=urban_road,
    parameter="P",
)

riskmap.add_diffuse_info_to_map(
    info_col_name="urban_build_P_load",
    load_method=urban_buildings,
    parameter="P",
)

riskmap.add_diffuse_info_to_map(
    info_col_name="urban_road_NH4_load",
    load_method=urban_road,
    parameter="NH4",
)

riskmap.add_diffuse_info_to_map(
    info_col_name="urban_build_NH4_load",
    load_method=urban_buildings,
    parameter="NH4",
)

In [ ]:
from utils import consentsRisk

In [ ]:
riskmap.add_all_risk(
    [
        {
            "type": "point",
            "info_col": "csoInfo",
            "risk_name": "cso_P_Risk",
            "risk_method": consentsRisk.add_cso_risk("P"),
            "distance_scaling": P_values.p_distance_scaling,
        },
        {
            "type": "point",
            "info_col": "wwtwInfo",
            "risk_name": "wwtw_P_Risk",
            "risk_method": consentsRisk.add_wwtw_risk("P"),
            "distance_scaling": P_values.p_distance_scaling,
        },
        {
            "type": "diffuse",
            "info_col": "agri_P_load",
            "risk_name": "agri_P_Risk",
            "risk_method": lambda x: x / P_values.P_GUIDELINE * 100,
            "distance_scaling": P_values.p_distance_scaling,
        },
        {
            "type": "diffuse",
            "info_col": "livestock_P_load",
            "risk_name": "livestock_P_Risk",
            "risk_method": lambda x: x / P_values.P_GUIDELINE * 100,
            "distance_scaling": P_values.p_distance_scaling,
        },
        {
            "type": "diffuse",
            "info_col": "urban_road_P_load",
            "risk_name": "urban_road_P_Risk",
            "risk_method": lambda x: x / P_values.P_GUIDELINE * 100,
            "distance_scaling": P_values.p_distance_scaling,
        },
        {
            "type": "diffuse",
            "info_col": "urban_build_P_load",
            "risk_name": "urban_build_P_Risk",
            "risk_method": lambda x: x / P_values.P_GUIDELINE * 100,
            "distance_scaling": P_values.p_distance_scaling,
        },
    ],
    risk_name="P_Risk",
)

In [ ]:
riskmap.add_all_risk(
    [
        {
            "type": "point",
            "info_col": "csoInfo",
            "risk_name": "cso_NH4_Risk",
            "risk_method": consentsRisk.add_cso_risk("NH4"),
            "distance_scaling": NH4_values.nh4_distance_scaling,
        },
        {
            "type": "point",
            "info_col": "wwtwInfo",
            "risk_name": "wwtw_NH4_Risk",
            "risk_method": consentsRisk.add_wwtw_risk("NH4"),
            "distance_scaling": NH4_values.nh4_distance_scaling,
        },
        {
            "type": "diffuse",
            "info_col": "agri_NH4_load",
            "risk_name": "agri_NH4_Risk",
            "risk_method": lambda x: x / NH4_values.NH4_GUIDELINE * 100,
            "distance_scaling": NH4_values.nh4_distance_scaling,
        },
        {
            "type": "diffuse",
            "info_col": "livestock_NH4_load",
            "risk_name": "livestock_NH4_Risk",
            "risk_method": lambda x: x / NH4_values.NH4_GUIDELINE * 100,
            "distance_scaling": NH4_values.nh4_distance_scaling,
        },
        {
            "type": "diffuse",
            "info_col": "urban_road_NH4_load",
            "risk_name": "urban_road_NH4_Risk",
            "risk_method": lambda x: x / NH4_values.NH4_GUIDELINE * 100,
            "distance_scaling": NH4_values.nh4_distance_scaling,
        },
        {
            "type": "diffuse",
            "info_col": "urban_build_NH4_load",
            "risk_name": "urban_build_NH4_Risk",
            "risk_method": lambda x: x / NH4_values.NH4_GUIDELINE * 100,
            "distance_scaling": NH4_values.nh4_distance_scaling,
        },
    ],
    risk_name="NH4_Risk",
)

In [ ]:
riskmap.add_all_risk(
    [
        {
            "type": "simple",
            "risk_name": "P_Risk",
        },
        {
            "type": "simple",
            "risk_name": "NH4_Risk",
        },
    ],
    risk_name="Risk",
    aggregation_method=np.mean,
)

In [ ]:
river_points = riskmap.get_risk_map()

## Explore

In [ ]:
# Join the values to the clipped dataframes. For WWTWs, there are multiple
# values for each WWTW, so we need to join on the DISCHARGE_SITE_TYPE_CODE as well.
wwtw = (
    wwtw.set_index("DISCHARGE_SITE_TYPE_CODE", append=True)
    .join(
        consentsRisk.get_wwtw_values(["P", "NH4"]).set_index(
            "DISCHARGE_SITE_TYPE_CODE", append=True
        ),
        how="left",
        validate="1:1",
    )
    .reset_index("DISCHARGE_SITE_TYPE_CODE")
)
cso = cso.join(
    consentsRisk.get_cso_values(["P", "NH4"]), how="left", validate="1:1"
)
cso["SpillDurationStr"] = cso["SpillDuration"].astype(str)

In [ ]:
csocols = [
    "geometry",
    "outfallLocation",
    "Site Name\n(EA Consents Database)",
    "SpillDurationStr",
    "Counted spills using 12-24h count method",
    "P",
    "NH4",
    "Weir Setting",
]

wwtwcols = [
    "geometry",
    "outfallLocation",
    "DISCHARGE_SITE_NAME",
    "P",
    "NH4",
    "DWF",
]

csolines = cso.apply(
    lambda x: shapely.LineString(
        [x["geometry"], x["outfallLocation"]]
        if x["geometry"] is not None and x["outfallLocation"] is not None
        else None
    ),
    axis=1,
).set_crs(cso.crs)
wwtwlines = wwtw.apply(
    lambda x: shapely.LineString(
        [x["geometry"], x["outfallLocation"]]
        if x["geometry"] is not None and x["outfallLocation"] is not None
        else None
    ),
    axis=1,
).set_crs(cso.crs)

In [ ]:
import branca

j, _, _ = riskmap.get_joined_catchments()
# Plot the catchment
m = j.explore(
    highlight=False,
    tooltip=False,
    popup=False,
    color="grey",
)

# Plot the rivers
rivers.explore(
    m=m,
    color="blue",
)
riskmap.get_rivers_split().explore(
    m=m, color="darkblue", style_kwds={"dashArray": "5, 5"}
)

# Plot the risk
river_points[
    [
        "geometry",
    ]
    + [col for col in river_points.columns if col.endswith("Risk")]
].explore(
    m=m,
    column="Risk",
    cmap="RdYlGn_r",
    vmax=100,
    marker_kwds={"radius": 5},
    legend_kwds={
        "caption": "Risk (0-100)",
        "color": "black",
        "fontsize": 12,
        "transparent": False,
    },
)

# Plot the CSOs, WwTWs and connecting lines
cso[csocols].explore(m=m, color="pink", marker_kwds={"radius": 8})
wwtw[wwtwcols].explore(m=m, color="yellow", marker_kwds={"radius": 8})

cso[csocols].set_geometry("outfallLocation").explore(
    m=m, color="red", marker_kwds={"radius": 8}
)
wwtw[wwtwcols].set_geometry("outfallLocation").explore(
    m=m, color="brown", marker_kwds={"radius": 8}
)

csolines.explore(m=m, style_kwds={"color": "red", "dashArray": "5, 5"})
wwtwlines.explore(m=m, style_kwds={"color": "brown", "dashArray": "5, 5"})

legend_html = """
{% macro html(this, kwargs) %}
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 200px; height: 150px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.85;">
     &nbsp; <b>Legend</b> <br>
     &nbsp; CSO location &nbsp; <i class="fa fa-circle" style="color:pink"></i><br>
     &nbsp; CSO outfall location &nbsp; <i class="fa fa-circle" style="color:red"></i><br>
     &nbsp; WwTW location &nbsp; <i class="fa fa-circle" style="color:yellow"></i><br>
     &nbsp; WwTW outfall location &nbsp; <i class="fa fa-circle" style="color:brown"></i><br>
</div>
{% endmacro %}
"""

legend = branca.element.MacroElement()
legend._template = branca.element.Template(legend_html)

# Add the legend to the map
m.get_root().add_child(legend)

m